# Radius sensitivity


In [ ]:
from pathlib import Path
import sys
import os
import gc
import random
from time import perf_counter

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

REPO = next(p for p in (Path.cwd(), Path.cwd()/"classix_tanimoto_manhattan", *Path.cwd().parents) if (p/'src/classix_t.py').is_file())
sys.path[:0] = [str(REPO/'src'), str(REPO/'vendor/bitbirch')]

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from threadpoolctl import threadpool_limits
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from bitbirch import bitbirch
import bblean
import classix_t
from classix_t import CLASSIX_T

bitbirch.set_merge('diameter')
classix_t.print = lambda *args, **kwargs: None
classix_t.tqdm = lambda iterable, *args, **kwargs: iterable

In [ ]:
SEED = 101
NUM_CLUSTERS = 10
N_PER_CLUSTER = 1000
DIMENSION = 1024
POPCOUNTS = [50, 300]
FLIP_PROBABILITY = 0.3
RADII = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
MIN_PTS = 1
BRANCHING_FACTOR = 50

OUTPUT_DIR = REPO/'experiments/experiment_1/_generated/radius_sensitivity'
PLOT_DIR = REPO/'experiments/experiment_1/plots/synthetic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## Data


In [ ]:
def generate_data(num_clusters, pops, d, n, flip_prob=0.1, seed=None):
    """Generate the original seeded binary clusters; n is points per cluster."""
    assert len(pops)>1, "You must provide at least 2 cluster mean populations in order to generate meaningful data"
    if len(pops) != num_clusters:
        try:
            print("The number of mean populations is not equal to the number of clusters. Trying to generate mean populations keeping in mind the minimum and maximum pop counts provided by the user")
            pops = np.linspace(min(pops), max(pops), num_clusters)
        except:
            raise ValueError("The number of cluster centers must be equal to the number of cluster populations")

    size_flag = [pops[x]<d for x in range(len(pops))]

    assert all(size_flag), "The number of dimensions must be greater than the pop counts of the clusters"

    clustc = []
    data = []
    if seed != None:
        np.random.seed(seed)
        random.seed(seed)

    labels = []

    for i in range(num_clusters):
        pop = pops[i]
        p1 = pop/d
        clustc.append(np.random.choice([0, 1], size=d, p=[1-p1, p1]))
        data.append(clustc[-1])
        labels.append(i)
        for j in range(n-1):
            labels.append(i)
            # Instead of flipping exactly pop*flip_prob bits, we flip each bit with a probability of flip_prob.
            flip_indices = np.random.choice([0, 1], size = d, p = [1-flip_prob*p1, flip_prob*p1])
            new_sample = np.logical_xor(clustc[-1], flip_indices).astype(np.int32)
            data.append(new_sample)
    return np.array(data).astype(np.int32), np.array(labels).astype(np.int32)

In [ ]:
data, truth = generate_data(NUM_CLUSTERS, POPCOUNTS, DIMENSION, N_PER_CLUSTER,
                            flip_prob=FLIP_PROBABILITY, seed=SEED)
permutation = np.arange(len(data))
np.random.shuffle(permutation)
data, truth = np.ascontiguousarray(data[permutation]), truth[permutation]

# Each representation is prepared before the clustering timer starts.
packed = np.packbits(data, axis=1, bitorder='little')
packed_big = np.packbits(data, axis=1, bitorder='big')
data_int64 = data.astype(np.int64)
fingerprints = [DataStructs.CreateFromBinaryText(row.tobytes()) for row in packed]
print(data.shape, np.unique(truth, return_counts=True)[1])

## Clustering


In [ ]:
rows = []
assignments = {}

def record(method, radius, labels, seconds):
    labels = np.asarray(labels, dtype=np.int64).copy()
    assignments[f'{method}_{radius:.1f}'] = labels
    rows.append(dict(radius=radius, method=method, seconds=seconds,
                     ari=float(adjusted_rand_score(truth, labels)),
                     ami=float(adjusted_mutual_info_score(truth, labels, average_method='arithmetic')),
                     clusters=int(np.unique(labels[labels >= 0]).size)))
    print(f'{method:16s} r={radius:.1f}  ARI={rows[-1]["ari"]:.4f}  AMI={rows[-1]["ami"]:.4f}')

# One measured fit per setting. Construction, fitting and label assignment are timed.
# These direct notebook timings do not use the former fresh-process isolation.
with threadpool_limits(limits=1):
    for radius in RADII:
        gc.collect()
        start = perf_counter()
        model = DBSCAN(eps=radius, min_samples=MIN_PTS, metric='jaccard', n_jobs=1)
        model.fit(data)
        labels = model.labels_
        seconds = perf_counter() - start
        record('dbscan', radius, labels, seconds)
        del model

        gc.collect()
        start = perf_counter()
        model = CLASSIX_T(radius=radius, minPts=MIN_PTS, mergeScale=1/np.sqrt(radius))
        model.fit(packed)
        labels = model.labels
        seconds = perf_counter() - start
        record('classix_packed', radius, labels, seconds)
        del model

        gc.collect()
        start = perf_counter()
        model = bitbirch.BitBirch(threshold=round(1-radius, 10), branching_factor=BRANCHING_FACTOR)
        model.fit(data_int64)
        labels = model.get_assignments(len(data))
        seconds = perf_counter() - start
        record('bitbirch', radius, labels, seconds)
        del model

        gc.collect()
        start = perf_counter()
        model = bblean.BitBirch(threshold=round(1-radius, 10), branching_factor=BRANCHING_FACTOR,
                               merge_criterion='diameter')
        model.fit(packed_big, input_is_packed=True, n_features=8*packed_big.shape[1])
        labels = model.get_assignments()
        seconds = perf_counter() - start
        record('bblean', radius, labels, seconds)
        del model

        gc.collect()
        start = perf_counter()
        distances = []
        for i in range(1, len(fingerprints)):
            similarities = DataStructs.BulkTanimotoSimilarity(fingerprints[i], fingerprints[:i])
            distances.extend([1-value for value in similarities])
        clusters = Butina.ClusterData(distances, len(fingerprints), radius,
                                      isDistData=True, reordering=False)
        labels = np.full(len(data), -1, dtype=np.int64)
        for cluster_id, members in enumerate(clusters):
            labels[list(members)] = cluster_id
        seconds = perf_counter() - start
        record('butina', radius, labels, seconds)
        del distances, clusters

import json
(OUTPUT_DIR/'scores.json').write_text(json.dumps(rows, indent=2)+'\n', encoding='utf-8')
np.savez_compressed(OUTPUT_DIR/'assignments.npz', truth=truth, **assignments)

## ARI and AMI


In [ ]:
METHODS = {
    'classix_packed': ('CLASSIX_T', 'red', 'x', '--'),
    'dbscan': ('DBSCAN', 'blue', 'o', '-.'),
    'bitbirch': ('BitBIRCH', 'green', 'v', ':'),
    'bblean': ('BitBIRCH-Lean', '#9467bd', 's', '--'),
    'butina': ('Taylor-Butina', '#e68613', 'D', '-'),
}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                    'axes.titlesize': 10, 'axes.labelsize': 9,
                    'xtick.labelsize': 9, 'ytick.labelsize': 9,
                    'axes.grid': False, 'svg.fonttype': 'none', 'pdf.fonttype': 42})
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.3), sharex=True, sharey=True)
fig.subplots_adjust(left=.075, right=.98, bottom=.18, top=.73, wspace=.15)
handles = []
for method, (label, color, marker, linestyle) in METHODS.items():
    selected = sorted((row for row in rows if row['method'] == method), key=lambda row: row['radius'])
    x = [row['radius'] for row in selected]
    for ax, metric in zip(axes, ('ari', 'ami')):
        ax.plot(x, [row[metric] for row in selected], label=label,
                color=color, marker=marker, linestyle=linestyle,
                linewidth=1.4, markersize=5, markerfacecolor='none', markeredgewidth=1.1)
    handles.append(Line2D([0], [0], label=label, color=color, marker=marker, linestyle=linestyle,
                          linewidth=1.4, markersize=5, markerfacecolor='none'))
for ax, metric in zip(axes, ('ARI', 'AMI')):
    ax.set_title(metric, pad=5)
    ax.set_xlabel('Distance threshold r')
    ax.set_xlim(.28, .82)
    ax.set_xticks([.3, .4, .5, .6, .7, .8])
    ax.set_ylim(-.04, 1.04)
    ax.set_yticks([0, .25, .5, .75, 1])
fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(.065, .995),
           ncol=3, fontsize=8, frameon=False, columnspacing=1.7, handlelength=2.4)
for extension in ('pdf', 'png', 'svg'):
    fig.savefig(PLOT_DIR/f'radius_ari_ami.{extension}', dpi=300, facecolor='white')
plt.show()